In [1]:
!pip install transformers accelerate scikit-learn pandas openpyxl -q

In [2]:
import os
import random
import pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
file_path = "/content/5K_HateSpeechAllGender.xlsx"

In [5]:
df = pd.read_excel(file_path, header=None)

print("Original shape:", df.shape)
print(df.head())

Original shape: (5320, 24)
                                                  0     1   \
0  S K  யாருடா நீ எனக்கே என்ன பாக்கனும் போல இருக்...  HATE   
1  டேய் உன்னோட சின்ன தங்கச்சி கொத்தும் கொலையுமா ந...  HATE   
2  பிரபல நாட்டாமை சுண்ணி ஊம்ப காசு அதிகம் வேணுமா?...  HATE   
3           அது ஒரு பொறம்போக்கு சிந்தனை கொண்ட ஜந்து.  HATE   
4   என்னோட சுன்னிய சப்பிட்டு தான் உன் குண்டியை கா...  HATE   

                    2         3   \
0     Hate against Men  IMPLICIT   
1  Hate against LGBTQ+  EXPLICIT   
2  Hate against LGBTQ+  EXPLICIT   
3   No Target- Profane  EXPLICIT   
4  Hate against LGBTQ+  EXPLICIT   

                                              4           5    6    7    8   \
0  ImplicitMen-Undermining / Mocking Masculinity  Religional  NaN  NaN  NaN   
1                 ExplicitLGBTQ-Homophobic Slurs         NaN  NaN  NaN  NaN   
2                 ExplicitLGBTQ-Homophobic Slurs         Nil   No  NaN  NaN   
3                                            Nil         Nil   No  

In [6]:
df = df.iloc[:, :6]

df.columns = [
    "Text",
    "Level 1 - Class",
    "Level 2 - Target",
    "Level 3 - Category",
    "Level 4 - Subcategory",
    "Level 5 - Bias"
]

TEXT_COL = "Text"

LEVEL1_COL = "Level 1 - Class"
LEVEL2_COL = "Level 2 - Target"
LEVEL3_COL = "Level 3 - Category"
LEVEL4_COL = "Level 4 - Subcategory"
LEVEL5_COL = "Level 5 - Bias"

label_cols = [
    LEVEL1_COL,
    LEVEL2_COL,
    LEVEL3_COL,
    LEVEL4_COL,
    LEVEL5_COL
]

print(df.head())

                                                Text Level 1 - Class  \
0  S K  யாருடா நீ எனக்கே என்ன பாக்கனும் போல இருக்...            HATE   
1  டேய் உன்னோட சின்ன தங்கச்சி கொத்தும் கொலையுமா ந...            HATE   
2  பிரபல நாட்டாமை சுண்ணி ஊம்ப காசு அதிகம் வேணுமா?...            HATE   
3           அது ஒரு பொறம்போக்கு சிந்தனை கொண்ட ஜந்து.            HATE   
4   என்னோட சுன்னிய சப்பிட்டு தான் உன் குண்டியை கா...            HATE   

      Level 2 - Target Level 3 - Category  \
0     Hate against Men           IMPLICIT   
1  Hate against LGBTQ+           EXPLICIT   
2  Hate against LGBTQ+           EXPLICIT   
3   No Target- Profane           EXPLICIT   
4  Hate against LGBTQ+           EXPLICIT   

                           Level 4 - Subcategory Level 5 - Bias  
0  ImplicitMen-Undermining / Mocking Masculinity     Religional  
1                 ExplicitLGBTQ-Homophobic Slurs            NaN  
2                 ExplicitLGBTQ-Homophobic Slurs            Nil  
3                               

In [7]:
df = df.copy()

df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()

for col in label_cols:
    df[col] = df[col].fillna("Nil")
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({
        "": "Nil",
        "nan": "Nil",
        "NaN": "Nil",
        "None": "Nil",
        "NONE": "Nil"
    })

# Remove accidental header rows if present inside data
df = df[df[TEXT_COL].str.lower() != "text"]

for col in label_cols:
    df = df[~df[col].str.lower().str.contains("level", na=False)]

# Keep only valid Level 1 labels
df = df[df[LEVEL1_COL].isin(["HATE", "NON-HATE"])]

# Remove empty text rows
df = df[df[TEXT_COL].str.strip() != ""]

df = df.reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)

for col in label_cols:
    print("\n" + "=" * 70)
    print(col)
    print(df[col].value_counts())

Cleaned dataset shape: (5308, 6)

Level 1 - Class
Level 1 - Class
HATE        2789
NON-HATE    2519
Name: count, dtype: int64

Level 2 - Target
Level 2 - Target
Nil                    2557
Hate against Women     1272
Hate against Men        774
No Target- Profane      512
Hate against LGBTQ+     183
Men- no profane          10
Name: count, dtype: int64

Level 3 - Category
Level 3 - Category
Nil         2531
IMPLICIT    1578
EXPLICIT    1199
Name: count, dtype: int64

Level 4 - Subcategory
Level 4 - Subcategory
Nil                                              3084
ExplicitMen-Verbal Abusive                        408
Sarcasm                                           367
ExplicitWomen-Verbal Abusive                      302
ImplicitWomen-Character assassination             213
ImplicitWomen-Slut-shaming                        128
ImplicitWomen-Dismissal                            99
ImplicitWomen-Body shaming                         95
ImplicitMen-Character assassination                8

In [8]:
label_encoders = {}
num_labels = {}

for col in label_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    label_encoders[col] = le
    num_labels[col] = len(le.classes_)

    print("\n" + "=" * 70)
    print(col)
    print("Classes:", list(le.classes_))
    print("Number of classes:", num_labels[col])

enc_label_cols = [col + "_enc" for col in label_cols]

print("\nEncoded columns:")
print(enc_label_cols)

print("\nAll columns:")
print(df.columns.tolist())


Level 1 - Class
Classes: ['HATE', 'NON-HATE']
Number of classes: 2

Level 2 - Target
Classes: ['Hate against LGBTQ+', 'Hate against Men', 'Hate against Women', 'Men- no profane', 'Nil', 'No Target- Profane']
Number of classes: 6

Level 3 - Category
Classes: ['EXPLICIT', 'IMPLICIT', 'Nil']
Number of classes: 3

Level 4 - Subcategory
Classes: ['ExplicitLGBTQ-Homophobic Slurs', 'ExplicitLGBTQ-Physical / Sexual Threats', 'ExplicitLGBTQ-Transphobic Attacks', 'ExplicitLGBTQ-Verbal Abusive', 'ExplicitMen-Masculinity Attacks', 'ExplicitMen-Verbal Abusive', 'ExplicitWomen-Sexual Objectification', 'ExplicitWomen-Sexual Threat', 'ExplicitWomen-Verbal Abusive', 'ImplicitLGBTQ-Character assassination', 'ImplicitLGBTQ-Exclusion', 'ImplicitLGBTQ-Mocking Identity', 'ImplicitLGBTQ-Moral policing', 'ImplicitLGBTQ-Pathologizing', 'ImplicitLGBTQ-Slut-shaming', 'ImplicitLGBTQ-Victim Blaming', 'ImplicitMen-Character assassination', 'ImplicitMen-Mocking Identity', 'ImplicitMen-Moral policing', 'ImplicitMen-

In [9]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df[LEVEL1_COL + "_enc"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df[LEVEL1_COL + "_enc"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (4246, 11)
Validation: (531, 11)
Test: (531, 11)


In [10]:
MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [11]:
class HateSpeechDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, text_col, label_cols):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.text_col = text_col
        self.label_cols = label_cols

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = str(row[self.text_col])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        labels = {}
        for col in self.label_cols:
            labels[col] = torch.tensor(int(row[col]), dtype=torch.long)

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels
        }

In [12]:
MAX_LEN = 128
BATCH_SIZE = 16

train_dataset = HateSpeechDataset(
    train_df,
    tokenizer,
    MAX_LEN,
    TEXT_COL,
    enc_label_cols
)

val_dataset = HateSpeechDataset(
    val_df,
    tokenizer,
    MAX_LEN,
    TEXT_COL,
    enc_label_cols
)

test_dataset = HateSpeechDataset(
    test_df,
    tokenizer,
    MAX_LEN,
    TEXT_COL,
    enc_label_cols
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [13]:
class HierarchicalMuRIL(nn.Module):
    def __init__(
        self,
        model_name,
        num_l1,
        num_l2,
        num_l3,
        num_l4,
        num_l5,
        dropout_rate=0.3
    ):
        super(HierarchicalMuRIL, self).__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.dropout = nn.Dropout(dropout_rate)

        self.level1_head = nn.Linear(hidden_size, num_l1)
        self.level2_head = nn.Linear(hidden_size + num_l1, num_l2)
        self.level3_head = nn.Linear(hidden_size + num_l1 + num_l2, num_l3)
        self.level4_head = nn.Linear(hidden_size + num_l1 + num_l2 + num_l3, num_l4)
        self.level5_head = nn.Linear(hidden_size + num_l1 + num_l2 + num_l3 + num_l4, num_l5)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits_l1 = self.level1_head(cls_output)
        prob_l1 = torch.softmax(logits_l1, dim=1)

        input_l2 = torch.cat([cls_output, prob_l1], dim=1)
        logits_l2 = self.level2_head(input_l2)
        prob_l2 = torch.softmax(logits_l2, dim=1)

        input_l3 = torch.cat([cls_output, prob_l1, prob_l2], dim=1)
        logits_l3 = self.level3_head(input_l3)
        prob_l3 = torch.softmax(logits_l3, dim=1)

        input_l4 = torch.cat([cls_output, prob_l1, prob_l2, prob_l3], dim=1)
        logits_l4 = self.level4_head(input_l4)
        prob_l4 = torch.softmax(logits_l4, dim=1)

        input_l5 = torch.cat([cls_output, prob_l1, prob_l2, prob_l3, prob_l4], dim=1)
        logits_l5 = self.level5_head(input_l5)

        return {
            "level1": logits_l1,
            "level2": logits_l2,
            "level3": logits_l3,
            "level4": logits_l4,
            "level5": logits_l5
        }

In [14]:
model = HierarchicalMuRIL(
    model_name=MODEL_NAME,
    num_l1=num_labels[LEVEL1_COL],
    num_l2=num_labels[LEVEL2_COL],
    num_l3=num_labels[LEVEL3_COL],
    num_l4=num_labels[LEVEL4_COL],
    num_l5=num_labels[LEVEL5_COL],
    dropout_rate=0.3
)

model = model.to(device)

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
def make_class_weights(dataframe, label_col, num_classes):
    counts = dataframe[label_col].value_counts().sort_index()

    full_counts = np.zeros(num_classes)

    for class_id, count in counts.items():
        full_counts[int(class_id)] = count

    # Avoid division by zero
    full_counts = np.where(full_counts == 0, 1, full_counts)

    # Smooth inverse-frequency weighting
    weights = 1.0 / np.sqrt(full_counts)

    # Normalize weights
    weights = weights / weights.sum() * num_classes

    return torch.tensor(weights, dtype=torch.float).to(device)

In [16]:
criterion_l1 = nn.CrossEntropyLoss(
    weight=make_class_weights(train_df, LEVEL1_COL + "_enc", num_labels[LEVEL1_COL])
)

criterion_l2 = nn.CrossEntropyLoss(
    weight=make_class_weights(train_df, LEVEL2_COL + "_enc", num_labels[LEVEL2_COL])
)

criterion_l3 = nn.CrossEntropyLoss(
    weight=make_class_weights(train_df, LEVEL3_COL + "_enc", num_labels[LEVEL3_COL])
)

criterion_l4 = nn.CrossEntropyLoss(
    weight=make_class_weights(train_df, LEVEL4_COL + "_enc", num_labels[LEVEL4_COL])
)

criterion_l5 = nn.CrossEntropyLoss(
    weight=make_class_weights(train_df, LEVEL5_COL + "_enc", num_labels[LEVEL5_COL])
)

In [17]:
print("Level 1 weights:", criterion_l1.weight)
print("Level 2 weights:", criterion_l2.weight)
print("Level 3 weights:", criterion_l3.weight)
print("Level 4 weights:", criterion_l4.weight)
print("Level 5 weights:", criterion_l5.weight)

Level 1 weights: tensor([0.9745, 1.0255], device='cuda:0')
Level 2 weights: tensor([0.8475, 0.4162, 0.3281, 3.6696, 0.2292, 0.5095], device='cuda:0')
Level 3 weights: tensor([1.1673, 1.0252, 0.8075], device='cuda:0')
Level 4 weights: tensor([1.0891, 1.4408, 1.2287, 1.0188, 0.6288, 0.2217, 0.6075, 0.8893, 0.2664,
        1.1764, 2.3528, 1.2287, 0.7701, 2.0376, 1.6637, 4.0752, 0.4906, 0.8150,
        1.4408, 0.9605, 1.2287, 1.2287, 0.4675, 0.3046, 0.4737, 1.0188, 0.7701,
        0.4181, 0.7843, 0.5822, 0.0819, 0.2385], device='cuda:0')
Level 5 weights: tensor([1.4914, 0.5700, 0.3659, 0.8723, 0.0875, 3.1047, 0.5081],
       device='cuda:0')


In [18]:
EPOCHS = 10
LEARNING_RATE = 2e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [19]:
def train_one_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()

    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        labels_l1 = batch["labels"][LEVEL1_COL + "_enc"].to(device)
        labels_l2 = batch["labels"][LEVEL2_COL + "_enc"].to(device)
        labels_l3 = batch["labels"][LEVEL3_COL + "_enc"].to(device)
        labels_l4 = batch["labels"][LEVEL4_COL + "_enc"].to(device)
        labels_l5 = batch["labels"][LEVEL5_COL + "_enc"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss_l1 = criterion_l1(outputs["level1"], labels_l1)
        loss_l2 = criterion_l2(outputs["level2"], labels_l2)
        loss_l3 = criterion_l3(outputs["level3"], labels_l3)
        loss_l4 = criterion_l4(outputs["level4"], labels_l4)
        loss_l5 = criterion_l5(outputs["level5"], labels_l5)

        loss = (
            1.5 * loss_l1 +
            1.2 * loss_l2 +
            1.0 * loss_l3 +
            1.2 * loss_l4 +
            1.2 * loss_l5
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [20]:
def evaluate(model, dataloader, device):
    model.eval()

    total_loss = 0

    true_l1, pred_l1 = [], []
    true_l2, pred_l2 = [], []
    true_l3, pred_l3 = [], []
    true_l4, pred_l4 = [], []
    true_l5, pred_l5 = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            labels_l1 = batch["labels"][LEVEL1_COL + "_enc"].to(device)
            labels_l2 = batch["labels"][LEVEL2_COL + "_enc"].to(device)
            labels_l3 = batch["labels"][LEVEL3_COL + "_enc"].to(device)
            labels_l4 = batch["labels"][LEVEL4_COL + "_enc"].to(device)
            labels_l5 = batch["labels"][LEVEL5_COL + "_enc"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss_l1 = criterion_l1(outputs["level1"], labels_l1)
            loss_l2 = criterion_l2(outputs["level2"], labels_l2)
            loss_l3 = criterion_l3(outputs["level3"], labels_l3)
            loss_l4 = criterion_l4(outputs["level4"], labels_l4)
            loss_l5 = criterion_l5(outputs["level5"], labels_l5)

            loss = (
                1.5 * loss_l1 +
                1.2 * loss_l2 +
                1.0 * loss_l3 +
                1.2 * loss_l4 +
                1.2 * loss_l5
            )

            total_loss += loss.item()

            p1 = torch.argmax(outputs["level1"], dim=1)
            p2 = torch.argmax(outputs["level2"], dim=1)
            p3 = torch.argmax(outputs["level3"], dim=1)
            p4 = torch.argmax(outputs["level4"], dim=1)
            p5 = torch.argmax(outputs["level5"], dim=1)

            true_l1.extend(labels_l1.cpu().numpy())
            pred_l1.extend(p1.cpu().numpy())

            true_l2.extend(labels_l2.cpu().numpy())
            pred_l2.extend(p2.cpu().numpy())

            true_l3.extend(labels_l3.cpu().numpy())
            pred_l3.extend(p3.cpu().numpy())

            true_l4.extend(labels_l4.cpu().numpy())
            pred_l4.extend(p4.cpu().numpy())

            true_l5.extend(labels_l5.cpu().numpy())
            pred_l5.extend(p5.cpu().numpy())

    results = {
        "loss": total_loss / len(dataloader),

        "level1_acc": accuracy_score(true_l1, pred_l1),
        "level1_macro_f1": f1_score(true_l1, pred_l1, average="macro", zero_division=0),

        "level2_acc": accuracy_score(true_l2, pred_l2),
        "level2_macro_f1": f1_score(true_l2, pred_l2, average="macro", zero_division=0),

        "level3_acc": accuracy_score(true_l3, pred_l3),
        "level3_macro_f1": f1_score(true_l3, pred_l3, average="macro", zero_division=0),

        "level4_acc": accuracy_score(true_l4, pred_l4),
        "level4_macro_f1": f1_score(true_l4, pred_l4, average="macro", zero_division=0),

        "level5_acc": accuracy_score(true_l5, pred_l5),
        "level5_macro_f1": f1_score(true_l5, pred_l5, average="macro", zero_division=0),
    }

    predictions = {
        "true_l1": true_l1,
        "pred_l1": pred_l1,

        "true_l2": true_l2,
        "pred_l2": pred_l2,

        "true_l3": true_l3,
        "pred_l3": pred_l3,

        "true_l4": true_l4,
        "pred_l4": pred_l4,

        "true_l5": true_l5,
        "pred_l5": pred_l5,
    }

    return results, predictions

In [21]:
best_val_f1 = 0
best_model_path = "best_hierarchical_muril_weighted.pt"

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print("-" * 50)

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device
    )

    val_results, _ = evaluate(
        model,
        val_loader,
        device
    )

    print("Train Loss:", train_loss)
    print("Validation Loss:", val_results["loss"])

    print("Level 1 Macro-F1:", val_results["level1_macro_f1"])
    print("Level 2 Macro-F1:", val_results["level2_macro_f1"])
    print("Level 3 Macro-F1:", val_results["level3_macro_f1"])
    print("Level 4 Macro-F1:", val_results["level4_macro_f1"])
    print("Level 5 Macro-F1:", val_results["level5_macro_f1"])

    avg_macro_f1 = (
        val_results["level1_macro_f1"] +
        val_results["level2_macro_f1"] +
        val_results["level3_macro_f1"] +
        val_results["level4_macro_f1"] +
        val_results["level5_macro_f1"]
    ) / 5

    print("Average Macro-F1:", avg_macro_f1)

    if avg_macro_f1 > best_val_f1:
        best_val_f1 = avg_macro_f1
        torch.save(model.state_dict(), best_model_path)
        print("Best model saved.")


Epoch 1/10
--------------------------------------------------
Train Loss: 10.64375394806826
Validation Loss: 10.435356308432187
Level 1 Macro-F1: 0.6376251137397634
Level 2 Macro-F1: 0.16428081551218904
Level 3 Macro-F1: 0.34427284427284427
Level 4 Macro-F1: 0.02680672268907563
Level 5 Macro-F1: 0.13520408163265304
Average Macro-F1: 0.26163791556930505
Best model saved.

Epoch 2/10
--------------------------------------------------
Train Loss: 10.278074845335537
Validation Loss: 10.101774832781624
Level 1 Macro-F1: 0.7829824762669573
Level 2 Macro-F1: 0.24817579706384835
Level 3 Macro-F1: 0.44873043704136634
Level 4 Macro-F1: 0.040436485369245545
Level 5 Macro-F1: 0.13520408163265304
Average Macro-F1: 0.3311058554748141
Best model saved.

Epoch 3/10
--------------------------------------------------
Train Loss: 9.931620357628155
Validation Loss: 9.831653174232034
Level 1 Macro-F1: 0.7776003066700741
Level 2 Macro-F1: 0.2607003245406408
Level 3 Macro-F1: 0.5712532052869838
Level 4 Macr

In [22]:
model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

model = model.to(device)
model.eval()

print("Best model loaded.")

Best model loaded.


In [23]:
test_results, test_predictions = evaluate(
    model,
    test_loader,
    device
)

print("\nTest Results")
print("=" * 80)

for key, value in test_results.items():
    print(f"{key}: {value}")


Test Results
loss: 8.988779096042409
level1_acc: 0.8060263653483992
level1_macro_f1: 0.8058032147764675
level2_acc: 0.615819209039548
level2_macro_f1: 0.29554554064318134
level3_acc: 0.6892655367231638
level3_macro_f1: 0.6645236893297809
level4_acc: 0.5310734463276836
level4_macro_f1: 0.039951745890523103
level5_acc: 0.903954802259887
level5_macro_f1: 0.13565069944891903


In [24]:
def print_report(level_name, true_values, pred_values, label_col):
    print("\n" + "=" * 100)
    print(level_name)
    print("=" * 100)

    target_names = label_encoders[label_col].classes_

    print(classification_report(
        true_values,
        pred_values,
        target_names=target_names,
        zero_division=0
    ))

In [26]:
import numpy as np

def print_report(level_name, true_values, pred_values, label_col):
    print("\n" + "=" * 100)
    print(level_name)
    print("=" * 100)

    target_names = label_encoders[label_col].classes_
    actual_labels = np.unique(true_values)

    print(classification_report(
        true_values,
        pred_values,
        labels=actual_labels, # Pass the actual labels present in the data
        target_names=target_names,
        zero_division=0
    ))

print_report(
    "Level 1: Hate / Non-Hate",
    test_predictions["true_l1"],
    test_predictions["pred_l1"],
    LEVEL1_COL
)

print_report(
    "Level 2: Target",
    test_predictions["true_l2"],
    test_predictions["pred_l2"],
    LEVEL2_COL
)

print_report(
    "Level 3: Explicit / Implicit / Nil",
    test_predictions["true_l3"],
    test_predictions["pred_l3"],
    LEVEL3_COL
)

print_report(
    "Level 4: Subcategory",
    test_predictions["true_l4"],
    test_predictions["pred_l4"],
    LEVEL4_COL
)

print_report(
    "Level 5: Bias",
    test_predictions["true_l5"],
    test_predictions["pred_l5"],
    LEVEL5_COL
)


Level 1: Hate / Non-Hate
              precision    recall  f1-score   support

        HATE       0.83      0.80      0.81       279
    NON-HATE       0.79      0.81      0.80       252

    accuracy                           0.81       531
   macro avg       0.81      0.81      0.81       531
weighted avg       0.81      0.81      0.81       531


Level 2: Target
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.00      0.00      0.00        15
   Hate against Men       0.44      0.50      0.47        84
 Hate against Women       0.45      0.57      0.50       137
    Men- no profane       0.00      0.00      0.00         2
                Nil       0.79      0.81      0.80       254
 No Target- Profane       0.00      0.00      0.00        39

           accuracy                           0.62       531
          macro avg       0.28      0.31      0.30       531
       weighted avg       0.56      0.62      0.59       531


Level 3: Explici

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2687: UserWarning: labels size, 27, does not match size of target_names, 32
  warnings.warn(


In [27]:
results_df = pd.DataFrame({
    "Text": test_df[TEXT_COL].values,

    "True_Level1": label_encoders[LEVEL1_COL].inverse_transform(test_predictions["true_l1"]),
    "Pred_Level1": label_encoders[LEVEL1_COL].inverse_transform(test_predictions["pred_l1"]),

    "True_Level2": label_encoders[LEVEL2_COL].inverse_transform(test_predictions["true_l2"]),
    "Pred_Level2": label_encoders[LEVEL2_COL].inverse_transform(test_predictions["pred_l2"]),

    "True_Level3": label_encoders[LEVEL3_COL].inverse_transform(test_predictions["true_l3"]),
    "Pred_Level3": label_encoders[LEVEL3_COL].inverse_transform(test_predictions["pred_l3"]),

    "True_Level4": label_encoders[LEVEL4_COL].inverse_transform(test_predictions["true_l4"]),
    "Pred_Level4": label_encoders[LEVEL4_COL].inverse_transform(test_predictions["pred_l4"]),

    "True_Level5": label_encoders[LEVEL5_COL].inverse_transform(test_predictions["true_l5"]),
    "Pred_Level5": label_encoders[LEVEL5_COL].inverse_transform(test_predictions["pred_l5"]),
})

results_df.to_csv("hierarchical_muril_test_predictions.csv", index=False)

results_df.head()

,Text,True_Level1,Pred_Level1,True_Level2,Pred_Level2,True_Level3,Pred_Level3,True_Level4,Pred_Level4,True_Level5,Pred_Level5
0,ம் போதும் போதும் லிஸ்ட் ரொம்ப பெருசா போய்ட்டு ...,NON-HATE,NON-HATE,Nil,Nil,Nil,Nil,Nil,Nil,Nil,Nil
1,கண்டேட் காக மேடமே நிறைய படம் நடிச்சிருக்காங்க ...,NON-HATE,NON-HATE,Nil,Nil,Nil,Nil,Nil,Nil,Nil,Nil
2,மூடிகிட்டு போய்விடு உன்னைப் பற்றி சொன்னால் தமி...,HATE,HATE,No Target- Profane,Hate against Men,IMPLICIT,EXPLICIT,Nil,ExplicitMen-Verbal Abusive,Nil,Nil
3,நல்ல தமிழில் பேசும் தலைவி அம்மா. படித்ததோ மெட்...,NON-HATE,NON-HATE,Nil,Nil,Nil,Nil,Nil,Nil,Nil,Nil
4,kundu... kundu... kundu penna...,HATE,HATE,Hate against Women,Hate against Women,IMPLICIT,IMPLICIT,ImplicitWomen-Body shaming,ExplicitMen-Verbal Abusive,Nil,Nil


In [28]:
def predict_text(text, model, tokenizer, max_len=128):
    model.eval()

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    pred_l1 = torch.argmax(outputs["level1"], dim=1).cpu().item()
    pred_l2 = torch.argmax(outputs["level2"], dim=1).cpu().item()
    pred_l3 = torch.argmax(outputs["level3"], dim=1).cpu().item()
    pred_l4 = torch.argmax(outputs["level4"], dim=1).cpu().item()
    pred_l5 = torch.argmax(outputs["level5"], dim=1).cpu().item()

    result = {
        "Text": text,
        "Level 1 - Class": label_encoders[LEVEL1_COL].inverse_transform([pred_l1])[0],
        "Level 2 - Target": label_encoders[LEVEL2_COL].inverse_transform([pred_l2])[0],
        "Level 3 - Category": label_encoders[LEVEL3_COL].inverse_transform([pred_l3])[0],
        "Level 4 - Subcategory": label_encoders[LEVEL4_COL].inverse_transform([pred_l4])[0],
        "Level 5 - Bias": label_encoders[LEVEL5_COL].inverse_transform([pred_l5])[0],
    }

    return result

In [29]:
def apply_consistency_rules(result):
    level1 = str(result["Level 1 - Class"]).lower()

    if "non" in level1:
        result["Level 2 - Target"] = "Nil"
        result["Level 3 - Category"] = "Nil"
        result["Level 4 - Subcategory"] = "Nil"
        result["Level 5 - Bias"] = "Nil"

    return result

In [31]:
sample_text = "poda gay punda"

result = predict_text(
    sample_text,
    model,
    tokenizer,
    MAX_LEN
)

result = apply_consistency_rules(result)

result

{'Text': 'poda gay punda',
 'Level 1 - Class': 'HATE',
 'Level 2 - Target': 'Hate against Men',
 'Level 3 - Category': 'EXPLICIT',
 'Level 4 - Subcategory': 'ExplicitMen-Verbal Abusive',
 'Level 5 - Bias': 'Nil'}

In [35]:
sample_text = "punda nee padichi enna pudunga pora. padippu punda ellam ethukku poi velayapaaru."

result = predict_text(
    sample_text,
    model,
    tokenizer,
    MAX_LEN
)

result = apply_consistency_rules(result)

result

{'Text': 'punda nee padichi enna pudunga pora. padippu punda ellam ethukku poi velayapaaru.',
 'Level 1 - Class': 'HATE',
 'Level 2 - Target': 'Hate against Women',
 'Level 3 - Category': 'EXPLICIT',
 'Level 4 - Subcategory': 'ExplicitMen-Verbal Abusive',
 'Level 5 - Bias': 'Nil'}

In [36]:
save_dir = "hierarchical_muril_weighted_hatespeech"
os.makedirs(save_dir, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(save_dir, "model.pt")
)

tokenizer.save_pretrained(save_dir)

with open(os.path.join(save_dir, "label_encoders.pkl"), "wb") as f:
    pickle.dump(label_encoders, f)

with open(os.path.join(save_dir, "num_labels.pkl"), "wb") as f:
    pickle.dump(num_labels, f)

print("Model saved to:", save_dir)

Model saved to: hierarchical_muril_weighted_hatespeech


In [37]:
save_dir = "hierarchical_muril_weighted_hatespeech"

tokenizer = AutoTokenizer.from_pretrained(save_dir)

with open(os.path.join(save_dir, "label_encoders.pkl"), "rb") as f:
    label_encoders = pickle.load(f)

with open(os.path.join(save_dir, "num_labels.pkl"), "rb") as f:
    num_labels = pickle.load(f)

loaded_model = HierarchicalMuRIL(
    model_name=MODEL_NAME,
    num_l1=num_labels[LEVEL1_COL],
    num_l2=num_labels[LEVEL2_COL],
    num_l3=num_labels[LEVEL3_COL],
    num_l4=num_labels[LEVEL4_COL],
    num_l5=num_labels[LEVEL5_COL],
    dropout_rate=0.3
)

loaded_model.load_state_dict(
    torch.load(
        os.path.join(save_dir, "model.pt"),
        map_location=device
    )
)

loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Saved model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved model loaded.
